# Focused DBSCAN and logistic regression refinement

The previous DBSCAN search used 6 density settings and C=1000. Test a fixed local grid around its leading density setting: **radius factors 0.5,1,2**, **min_samples 3,5,8**, and **C=100,1000,10000**. That is 9 label datasets and 27 classifier configurations. This is a local search, not an exhaustive optimization.

Keep 150 embeddings, the same training-only scalers, partial-label confidence rules, generated-label influence budgets, normalized fitting weights, threshold 0.5, nonempty fallback, and 78 validation trials fixed. No added features, PCA, or Kaggle submission. The central setting must reproduce the current validation reference (0.9103). Save every dataset/model/result, and prepare a new candidate only if the primary validation score improves.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from threadpoolctl import threadpool_limits

SEED = 42
REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
FEATURES = [f"emb_{i}" for i in range(150)]
LABEL_RULE = {'minimum_support': 3.0, 'prior_strength': 2.0, 'positive_threshold': 0.6, 'negative_threshold': 0.1, 'maximum_generated_weight': 0.25, 'maximum_class_fraction': 0.5}
RADIUS_FACTORS = [0.5, 1.0, 2.0]
MIN_SAMPLES_GRID = [3, 5, 8]
LOGISTIC_C_GRID = [100.0, 1000.0, 10000.0]
from sklearn.cluster import DBSCAN
cwd = Path.cwd()
DATA = next(q for q in [cwd, cwd/"part1", cwd/"AML-Midterm"/"part1"]
            if (q/"part1_train.csv").is_file() or (q/"data"/"part1_train.csv").is_file()).resolve()
INPUT_DIR = DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
OUT = DATA/"experiments"/"dbscan_refinement"
for folder in ["datasets", "label_audits", "clusters", "clustering_models", "classifier_models", "validation_predictions", "submissions", "labeling_models"]:
    (OUT/folder).mkdir(parents=True, exist_ok=True)
train = pd.read_csv(INPUT_DIR/"part1_train.csv")
valid = pd.read_csv(INPUT_DIR/"part1_val.csv")
X = train[FEATURES].to_numpy()
X_valid = valid[FEATURES].to_numpy()
truth = valid[REASONS].to_numpy(dtype=int)
original_y = train[REASONS].to_numpy(dtype=int)
known = train[REASONS].isin([0,1]).all(axis=1).to_numpy()
assert known.sum()==157 and len(train)==1171 and len(valid)==78
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isfinite(X).all() and np.isfinite(X_valid).all()
assert np.isin(truth,[0,1]).all()
baseline = joblib.load(DATA/"experiments/kmeans/models/baseline_C1000.joblib")
legacy_packet = joblib.load(DATA/"experiments/kmeans/models/k02.joblib")
classifier_scaler = baseline.named_steps["scaler"]
cluster_scaler = legacy_packet["kmeans_scaler"]
assert classifier_scaler.n_samples_seen_==157 and cluster_scaler.n_samples_seen_==1171
XS = classifier_scaler.transform(X)
VS = classifier_scaler.transform(X_valid)
XC = cluster_scaler.transform(X)
protected_paths = [INPUT_DIR/"part1_train.csv", INPUT_DIR/"part1_val.csv"] + sorted(DATA.glob("*submission.csv")) + [
    DATA/"experiments/kmeans/models/k02.joblib", DATA/"experiments/kmeans/datasets/k02_train.csv"
]
protected_hashes = {str(q.relative_to(DATA)):hashlib.sha256(q.read_bytes()).hexdigest() for q in protected_paths}
print("Using 150 features; 157 known-label and 1,014 unlabeled training trials; 78 validation trials.")

Using 150 features; 157 known-label and 1,014 unlabeled training trials; 78 validation trials.


## Reuse the existing controls

Known labels remain unchanged, unknown labels remain -100, and each reason is fitted on its own accepted targets. Generated weights are bounded as before. Logistic fitting weight totals remain 157 per reason, holding effective regularization comparable across label datasets. The fixed feature scaler was learned from the 157 known-label trials.

In [2]:
def label_predictions(probabilities):
    probabilities = np.asarray(probabilities)
    assert probabilities.shape[1]==5 and np.isfinite(probabilities).all()
    assert ((probabilities>=0)&(probabilities<=1)).all()
    prediction=(probabilities>=0.5).astype(int)
    empty=prediction.sum(axis=1)==0
    ids=np.flatnonzero(empty)
    prediction[ids, probabilities[ids].argmax(axis=1)]=1
    return prediction,empty


def score_probabilities(probabilities):
    predicted,empty=label_predictions(probabilities)
    return {"sample_f1":f1_score(truth,predicted,average="samples",zero_division=0),
            "macro_f1":f1_score(truth,predicted,average="macro",zero_division=0),
            "fallback_trials":int(empty.sum()),
            **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,predicted,average=None,zero_division=0)))},predicted,empty


def save_predictions(name, probabilities):
    metrics,predicted,empty=score_probabilities(probabilities)
    output=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):
        output[f"probability_{r}"]=probabilities[:,j]
        output[f"predicted_{r}"]=predicted[:,j]
    output["used_fallback"]=empty
    output.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    return metrics


DATASETS={}
LABEL_COUNTS=[]
FIT_CACHE={}


def save_dataset(name, targets, weights, probabilities=None, component=None, support=None):
    assert targets.shape==weights.shape==(len(train),5)
    np.testing.assert_array_equal(targets[known],original_y[known])
    np.testing.assert_array_equal(weights[known],np.ones((known.sum(),5)))
    assert np.isin(targets,[-100,0,1]).all() and (weights>=0).all()
    assert np.array_equal(weights>0,targets!=-100)
    frame=train[["ID"]+FEATURES].copy()
    audit=train[["ID"]].copy()
    for j,r in enumerate(REASONS):
        source=np.where(known,"original",np.where(weights[:,j]>0,"generated","unknown"))
        frame[r]=targets[:,j]
        frame[f"weight_{r}"]=weights[:,j]
        audit[r]=targets[:,j]
        audit[f"original_{r}"]=original_y[:,j]
        audit[f"weight_{r}"]=weights[:,j]
        audit[f"label_source_{r}"]=source
        if probabilities is not None:audit[f"estimated_probability_{r}"]=probabilities[:,j]
        pos=(~known)&(targets[:,j]==1)
        neg=(~known)&(targets[:,j]==0)
        LABEL_COUNTS.append({"dataset":name,"reason":r,
            "known_positives":int(original_y[known,j].sum()),
            "known_negatives":int((original_y[known,j]==0).sum()),
            "generated_positives":int(pos.sum()),"generated_negatives":int(neg.sum()),
            "generated_positive_weight":float(weights[pos,j].sum()),
            "generated_negative_weight":float(weights[neg,j].sum())})
    if component is not None:audit["component"]=component
    if support is not None:audit["effective_labeled_support"]=support
    frame["included_for_any_reason"]=(weights>0).any(axis=1)
    frame.to_csv(OUT/"datasets"/f"{name}.csv",index=False)
    audit.to_csv(OUT/"label_audits"/f"{name}.csv",index=False)
    DATASETS[name]={"targets":targets.copy(),"weights":weights.copy(),
                    "generated_trials":int(((weights>0).any(axis=1)&~known).sum())}
    return DATASETS[name]


def partial_labels(responsibilities):
    assert responsibilities.shape[0]==len(train)
    np.testing.assert_allclose(responsibilities.sum(axis=1),1,atol=1e-9)
    component_support=responsibilities[known].sum(axis=0)
    positives=responsibilities[known].T@original_y[known]
    prior=original_y[known].mean(axis=0)
    rate=(positives+LABEL_RULE["prior_strength"]*prior)/(component_support[:,None]+LABEL_RULE["prior_strength"])
    probabilities=responsibilities@rate
    support=responsibilities@component_support
    targets=np.full(original_y.shape,-100,dtype=int)
    weights=np.zeros(original_y.shape,dtype=float)
    targets[known]=original_y[known]
    weights[known]=1.0
    eligible=(~known)&(support>=LABEL_RULE["minimum_support"])
    for j in range(5):
        positive=eligible&(probabilities[:,j]>=LABEL_RULE["positive_threshold"])
        negative=eligible&(probabilities[:,j]<=LABEL_RULE["negative_threshold"])
        npos,nneg=int(positive.sum()),int(negative.sum())
        kp=int(original_y[known,j].sum());kn=int(known.sum()-kp)
        if not npos or not nneg:continue
        fraction=min(LABEL_RULE["maximum_class_fraction"],
                     LABEL_RULE["maximum_generated_weight"]*npos/kp,
                     LABEL_RULE["maximum_generated_weight"]*nneg/kn)
        targets[positive,j]=1;targets[negative,j]=0
        weights[positive,j]=fraction*kp/npos
        weights[negative,j]=fraction*kn/nneg
        assert weights[~known,j].max()<=LABEL_RULE["maximum_generated_weight"]+1e-12
        np.testing.assert_allclose(weights[positive,j].sum()/kp,weights[negative,j].sum()/kn,atol=1e-12)
    components=pd.DataFrame({"component":np.arange(len(component_support)),"effective_labeled_support":component_support,
                             "total_membership_mass":responsibilities.sum(axis=0)})
    for j,r in enumerate(REASONS):
        components[f"weighted_known_positive_{r}"]=positives[:,j]
        components[f"estimated_probability_{r}"]=rate[:,j]
    return targets,weights,probabilities,support,components


def fit_reason_models(dataset_name, kind="logistic", parameter=1000.0):
    dataset=DATASETS[dataset_name]
    estimators=[]
    for j in range(5):
        y=dataset["targets"][:,j];w=dataset["weights"][:,j]
        included=w>0
        assert np.isin(y[included],[0,1]).all() and included[known].all()
        key=(kind,float(parameter),j,hashlib.sha256(y.tobytes()+w.tobytes()).hexdigest())
        if key not in FIT_CACHE:
            if kind=="logistic":
                model=LogisticRegression(C=float(parameter),solver="lbfgs",max_iter=5000,tol=1e-4,random_state=SEED)
                normalized=w[included]*known.sum()/w[included].sum()
                with warnings.catch_warnings():
                    warnings.simplefilter("error",ConvergenceWarning)
                    model.fit(XS[included],y[included],sample_weight=normalized)
            else:
                model={"neighbors":NearestNeighbors(n_neighbors=min(int(parameter),int(included.sum())),algorithm="brute").fit(XS[included]),
                       "target":y[included].copy(),"sample_weight":w[included].copy()}
            FIT_CACHE[key]=model
        estimators.append(FIT_CACHE[key])
    return {"kind":kind,"parameter":float(parameter),"scaler":classifier_scaler,"estimators":estimators,
            "features":FEATURES,"reasons":REASONS,"dataset":dataset_name,
            "logistic_weight_normalization_total":int(known.sum())}


def predict_reason_models(model, raw_features):
    scaled=model["scaler"].transform(np.asarray(raw_features))
    output=[]
    for estimator in model["estimators"]:
        if model["kind"] in ["logistic", "rbf_svm", "gaussian_nb"]:
            positive=list(estimator.classes_).index(1)
            probability=estimator.predict_proba(scaled)[:,positive]
        else:
            distance,indices=estimator["neighbors"].kneighbors(scaled)
            zero=distance==0
            exact=zero.any(axis=1)
            distance_weight=np.divide(1.,distance,out=np.zeros_like(distance),where=distance!=0)
            distance_weight[exact]=zero[exact].astype(float)
            vote=distance_weight*estimator["sample_weight"][indices]
            probability=(vote*estimator["target"][indices]).sum(axis=1)/vote.sum(axis=1)
        output.append(probability)
    probabilities=np.column_stack(output)
    assert np.isfinite(probabilities).all()
    return np.clip(probabilities,0,1)


known_y=np.full_like(original_y,-100);known_y[known]=original_y[known]
known_w=np.zeros(original_y.shape);known_w[known]=1.0
save_dataset("known_only",known_y,known_w)
with threadpool_limits(limits=1):
    reproduced=fit_reason_models("known_only")
np.testing.assert_allclose(predict_reason_models(reproduced,X_valid),baseline.predict_proba(X_valid),rtol=1e-8,atol=1e-10)
reference_rows=[]
for name,model,kaggle in [("submitted_supervised",baseline,0.87072),("submitted_k02",legacy_packet["classifier"],0.87393)]:
    metrics=save_predictions(name,model.predict_proba(X_valid))
    reference_rows.append({"experiment":name,"dataset":"frozen reference","classifier":"logistic","parameter":1000.0,
                           "generated_trials":0 if name=="submitted_supervised" else 388,**metrics,"kaggle_score":kaggle})
print("Confirmed that the new per-reason trainer reproduces the supervised reference.")

Confirmed that the new per-reason trainer reproduces the supervised reference.


In [3]:
def targets_from_probabilities(probabilities,support,valid_probability=None):
    valid_probability=np.ones_like(probabilities,dtype=bool) if valid_probability is None else valid_probability
    targets=np.full_like(original_y,-100)
    weights=np.zeros(original_y.shape)
    targets[known]=original_y[known];weights[known]=1.0
    eligible=(~known)[:,None]&(np.asarray(support)>=LABEL_RULE["minimum_support"])[:,None]&valid_probability
    for j in range(5):
        positive=eligible[:,j]&(probabilities[:,j]>=LABEL_RULE["positive_threshold"])
        negative=eligible[:,j]&(probabilities[:,j]<=LABEL_RULE["negative_threshold"])
        np_,nn=int(positive.sum()),int(negative.sum())
        kp=int(original_y[known,j].sum());kn=int(known.sum()-kp)
        if not np_ or not nn:continue
        fraction=min(LABEL_RULE["maximum_class_fraction"],LABEL_RULE["maximum_generated_weight"]*np_/kp,
                     LABEL_RULE["maximum_generated_weight"]*nn/kn)
        targets[positive,j]=1;targets[negative,j]=0
        weights[positive,j]=fraction*kp/np_;weights[negative,j]=fraction*kn/nn
        assert weights[~known,j].max()<=0.25+1e-12
        np.testing.assert_allclose(weights[positive,j].sum()/kp,weights[negative,j].sum()/kn,atol=1e-12)
    return targets,weights




In [4]:
reference_dir=DATA/"experiments"/"additional_methods"
reference_record=json.loads((reference_dir/"submission_manifest.json").read_text())
reference_model=joblib.load(reference_dir/reference_record["model_file"])
reference_probability=predict_reason_models(reference_model,X_valid)
reference_metrics=save_predictions("dbscan_current_reference",reference_probability)
np.testing.assert_allclose(reference_metrics["sample_f1"],reference_record["validation_sample_f1"],atol=1e-14)
reference_density=joblib.load(reference_dir/"labeling_models/dbscan_m5_q25.joblib")
CENTER_RADIUS=float(reference_density["model"].eps)
reference_data=pd.read_csv(reference_dir/"datasets/dbscan_m5_q25.csv",float_precision="round_trip")
reference_targets=reference_data[REASONS].to_numpy(dtype=int)
reference_weights=reference_data[[f"weight_{r}" for r in REASONS]].to_numpy()
extra_protected=[reference_dir/reference_record["model_file"],reference_dir/"datasets/dbscan_m5_q25.csv",
                 reference_dir/"labeling_models/dbscan_m5_q25.joblib",reference_dir/reference_record["submission_file"]]
protected_hashes.update({str(q.relative_to(DATA)):hashlib.sha256(q.read_bytes()).hexdigest() for q in extra_protected})
REFINEMENT_ROWS=[]
REFINEMENT_MODELS={}
DENSITY_ROWS=[]


def run_local_density(min_samples):
    with threadpool_limits(limits=1):
        for radius_factor in RADIUS_FACTORS:
            name=f"local_m{min_samples}_r{str(radius_factor).replace('.','p')}"
            eps=CENTER_RADIUS*radius_factor
            density=DBSCAN(eps=eps,min_samples=min_samples,metric="euclidean",algorithm="brute",n_jobs=1).fit(XC)
            ids=density.labels_;clusters=np.unique(ids[ids>=0])
            if len(clusters):
                membership=np.column_stack([ids==c for c in clusters]).astype(float)
                support_by_cluster=membership[known].sum(axis=0)
                positive=membership[known].T@original_y[known]
                prior=original_y[known].mean(axis=0)
                rates=(positive+LABEL_RULE["prior_strength"]*prior)/(support_by_cluster[:,None]+LABEL_RULE["prior_strength"])
                probability=membership@rates
                support=membership@support_by_cluster
                details=pd.DataFrame({"cluster":clusters,"training_trials":membership.sum(axis=0),"known_support":support_by_cluster})
                for j,r in enumerate(REASONS):details[f"probability_{r}"]=rates[:,j]
            else:
                probability=np.tile(original_y[known].mean(axis=0),(len(train),1));support=np.zeros(len(train))
                details=pd.DataFrame(columns=["cluster","training_trials","known_support"])
            y,w=targets_from_probabilities(probability,support)
            audit_probability=np.where((ids>=0)[:,None],probability,np.nan)
            dataset=save_dataset(name,y,w,audit_probability,ids,support)
            assert (w[(ids<0)&~known]==0).all()
            details.to_csv(OUT/"clusters"/f"{name}.csv",index=False)
            joblib.dump({"model":density,"scaler":cluster_scaler,"label_rule":LABEL_RULE},
                        OUT/"labeling_models"/f"{name}.joblib",compress=3)
            is_central=min_samples==5 and radius_factor==1.0
            if is_central:
                np.testing.assert_array_equal(y,reference_targets)
                np.testing.assert_allclose(w,reference_weights,rtol=1e-12,atol=1e-14)
            DENSITY_ROWS.append({"dataset":name,"min_samples":min_samples,"radius_factor":radius_factor,"eps":eps,
                                 "clusters":len(clusters),"noise_trials":int((ids<0).sum()),"generated_trials":dataset["generated_trials"]})
            for C in LOGISTIC_C_GRID:
                model_name=f"{name}__logistic_C{int(C)}"
                model=fit_reason_models(name,"logistic",C)
                probabilities=predict_reason_models(model,X_valid)
                metrics=save_predictions(model_name,probabilities)
                if is_central and C==1000.0:
                    np.testing.assert_allclose(probabilities,reference_probability,rtol=1e-8,atol=1e-10)
                joblib.dump(model,OUT/"classifier_models"/f"{model_name}.joblib",compress=3)
                distance=abs(float(np.log2(radius_factor)))+abs(min_samples-5)+abs(float(np.log10(C/1000)))
                row={"experiment":model_name,"dataset":name,"min_samples":min_samples,"radius_factor":radius_factor,
                     "eps":eps,"C":C,"clusters":len(clusters),"noise_trials":int((ids<0).sum()),
                     "generated_trials":dataset["generated_trials"],"distance_from_original_settings":distance,
                     **metrics,"delta_vs_current_best":metrics["sample_f1"]-reference_metrics["sample_f1"],"kaggle_score":np.nan}
                REFINEMENT_ROWS.append(row);REFINEMENT_MODELS[model_name]=model
                print(f"{model_name}: sample F1={metrics['sample_f1']:.4f}, macro F1={metrics['macro_f1']:.4f}, "
                      f"change={row['delta_vs_current_best']:+.4f}",flush=True)
print("Center radius:",CENTER_RADIUS,"current F1:",reference_metrics["sample_f1"])

Center radius: 0.00012142366141394104 current F1: 0.9102564102564102


In [5]:
run_local_density(3)

local_m3_r0p5__logistic_C100: sample F1=0.8889, macro F1=0.7901, change=-0.0214


local_m3_r0p5__logistic_C1000: sample F1=0.8974, macro F1=0.8039, change=-0.0128


local_m3_r0p5__logistic_C10000: sample F1=0.8974, macro F1=0.8039, change=-0.0128


local_m3_r1p0__logistic_C100: sample F1=0.8889, macro F1=0.7901, change=-0.0214


local_m3_r1p0__logistic_C1000: sample F1=0.9103, macro F1=0.8114, change=+0.0000


local_m3_r1p0__logistic_C10000: sample F1=0.8974, macro F1=0.8039, change=-0.0128


local_m3_r2p0__logistic_C100: sample F1=0.8889, macro F1=0.7939, change=-0.0214


local_m3_r2p0__logistic_C1000: sample F1=0.8974, macro F1=0.8039, change=-0.0128


local_m3_r2p0__logistic_C10000: sample F1=0.8974, macro F1=0.8039, change=-0.0128


In [6]:
run_local_density(5)

local_m5_r0p5__logistic_C100: sample F1=0.8889, macro F1=0.7901, change=-0.0214


local_m5_r0p5__logistic_C1000: sample F1=0.8974, macro F1=0.8039, change=-0.0128


local_m5_r0p5__logistic_C10000: sample F1=0.8974, macro F1=0.8039, change=-0.0128


local_m5_r1p0__logistic_C100: sample F1=0.8889, macro F1=0.7901, change=-0.0214


local_m5_r1p0__logistic_C1000: sample F1=0.9103, macro F1=0.8114, change=+0.0000


local_m5_r1p0__logistic_C10000: sample F1=0.8974, macro F1=0.8039, change=-0.0128


local_m5_r2p0__logistic_C100: sample F1=0.8889, macro F1=0.7939, change=-0.0214


local_m5_r2p0__logistic_C1000: sample F1=0.8974, macro F1=0.8039, change=-0.0128


local_m5_r2p0__logistic_C10000: sample F1=0.8974, macro F1=0.8039, change=-0.0128


In [7]:
run_local_density(8)

local_m8_r0p5__logistic_C100: sample F1=0.8889, macro F1=0.7901, change=-0.0214


local_m8_r0p5__logistic_C1000: sample F1=0.8974, macro F1=0.8039, change=-0.0128


local_m8_r0p5__logistic_C10000: sample F1=0.8974, macro F1=0.8039, change=-0.0128


local_m8_r1p0__logistic_C100: sample F1=0.8889, macro F1=0.7901, change=-0.0214


local_m8_r1p0__logistic_C1000: sample F1=0.9103, macro F1=0.8114, change=+0.0000


local_m8_r1p0__logistic_C10000: sample F1=0.8974, macro F1=0.8039, change=-0.0128


local_m8_r2p0__logistic_C100: sample F1=0.8889, macro F1=0.7939, change=-0.0214


local_m8_r2p0__logistic_C1000: sample F1=0.8974, macro F1=0.8039, change=-0.0128


local_m8_r2p0__logistic_C10000: sample F1=0.8974, macro F1=0.8039, change=-0.0128


In [8]:
comparison=pd.DataFrame(REFINEMENT_ROWS)
comparison.to_csv(OUT/"comparison.csv",index=False)
pd.DataFrame(DENSITY_ROWS).to_csv(OUT/"density_comparison.csv",index=False)
pd.DataFrame(LABEL_COUNTS).to_csv(OUT/"label_counts.csv",index=False)
ranked=comparison.sort_values(["sample_f1","distance_from_original_settings","C","experiment"],
                              ascending=[False,True,True,True],kind="stable")
best=ranked.iloc[0]
print("Best local setting:",best.experiment)
display(ranked.head(12)[["min_samples","radius_factor","C","generated_trials","sample_f1","macro_f1","delta_vs_current_best"]].round(4))
print("Unique generated target/weight tables:",len({hashlib.sha256(d['targets'].tobytes()+d['weights'].tobytes()).hexdigest()
                                                   for name,d in DATASETS.items() if name.startswith('local_')}))

Best local setting: local_m5_r1p0__logistic_C1000


,min_samples,radius_factor,C,generated_trials,sample_f1,macro_f1,delta_vs_current_best
13,5,1.0,1000.0,133,0.9103,0.8114,0.0000
4,3,1.0,1000.0,133,0.9103,0.8114,0.0000
22,8,1.0,1000.0,133,0.9103,0.8114,0.0000
10,5,0.5,1000.0,81,0.8974,0.8039,-0.0128
16,5,2.0,1000.0,173,0.8974,0.8039,-0.0128
14,5,1.0,10000.0,133,0.8974,0.8039,-0.0128
11,5,0.5,10000.0,81,0.8974,0.8039,-0.0128
17,5,2.0,10000.0,173,0.8974,0.8039,-0.0128
1,3,0.5,1000.0,81,0.8974,0.8039,-0.0128
7,3,2.0,1000.0,173,0.8974,0.8039,-0.0128


Unique generated target/weight tables: 3


In [9]:
from sklearn.base import clone
selected=joblib.load(OUT/"classifier_models"/f"{best.experiment}.joblib")
np.testing.assert_allclose(predict_reason_models(selected,X_valid),predict_reason_models(REFINEMENT_MODELS[best.experiment],X_valid),rtol=1e-12,atol=1e-12)
frame=pd.read_csv(OUT/"datasets"/f"{best.dataset}.csv",float_precision="round_trip")
assert frame.ID.tolist()==train.ID.tolist()
replay={**selected,"estimators":[]}
scaled=classifier_scaler.transform(frame[FEATURES].to_numpy())
with threadpool_limits(limits=1):
    for j,r in enumerate(REASONS):
        y=frame[r].to_numpy(dtype=int);w=frame[f"weight_{r}"].to_numpy()
        included=w>0
        estimator=clone(selected["estimators"][j])
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            estimator.fit(scaled[included],y[included],sample_weight=w[included]*known.sum()/w[included].sum())
        replay["estimators"].append(estimator)
np.testing.assert_array_equal(label_predictions(predict_reason_models(replay,X_valid))[0],label_predictions(predict_reason_models(selected,X_valid))[0])
for relative,digest in protected_hashes.items():assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest()==digest

candidate=None
if best.sample_f1>reference_metrics["sample_f1"]+1e-12:
    test=pd.read_csv(INPUT_DIR/"part1_test.csv")
    sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    predicted,empty=label_predictions(predict_reason_models(selected,test[FEATURES].to_numpy()))
    output=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in predicted]})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    path=OUT/"submissions"/f"part1_dbscan_{best.experiment.replace('__','_')}_submission.csv"
    output.to_csv(path,index=False)
    pd.testing.assert_frame_equal(pd.read_csv(path),output)
    candidate={"experiment":best.experiment,"validation_sample_f1":float(best.sample_f1),"validation_macro_f1":float(best.macro_f1),
               "submission_file":str(path.relative_to(OUT)),"test_rows":len(output),
               "submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
               "model_file":f"classifier_models/{best.experiment}.joblib","kaggle_score":None,"kaggle_status":"prepared locally; not submitted"}
    (OUT/"submission_manifest.json").write_text(json.dumps(candidate,indent=2)+"\n")
settings={"features":FEATURES,"radius_center":CENTER_RADIUS,"radius_factors":RADIUS_FACTORS,
    "min_samples_grid":MIN_SAMPLES_GRID,"logistic_C_grid":LOGISTIC_C_GRID,"label_rule":LABEL_RULE,
    "classifier_scaler_fit_rows":157,"clustering_scaler_fit_rows":1171,"fitting_weight_total_per_reason":157,
    "threshold":0.5,"nonempty_fallback":True,"validation_rows":78,"validation_used_for_fit":False,
    "reference_sample_f1":reference_metrics["sample_f1"],"best_experiment":best.experiment,
    "best_sample_f1":float(best.sample_f1),"best_macro_f1":float(best.macro_f1),"candidate":candidate,
    "protected_input_sha256":protected_hashes,
    "dataset_sha256":{str(q.relative_to(OUT)):hashlib.sha256(q.read_bytes()).hexdigest() for q in sorted((OUT/"datasets").glob('*.csv'))},
    "reported_kaggle_best":0.87393,"new_kaggle_scores":None,
    "selection":"sample-wise F1; ties favor settings closest to original, then smaller C"}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
lines=["# Focused DBSCAN and logistic regression refinement","",
       "A fixed local grid evaluates 9 density settings and 3 classifier regularization strengths each, using only the same 150 embeddings. "
       "The previous DBSCAN + LR reference is sample-wise F1 0.9103 and macro F1 0.8114 on the same 78 validation trials.","",
       f"Radius center: **{CENTER_RADIUS:.17g}**. Radius factors: **0.5,1,2**. Min_samples: **3,5,8**. C: **100,1000,10000**.","",
       "| Min samples | Radius factor | C | Generated trials | Sample-wise F1 | Macro F1 | Change from reference |",
       "|---|---:|---:|---:|---:|---:|---:|"]
for row in ranked.itertuples(index=False):
    lines.append(f"| {row.min_samples} | {row.radius_factor:g} | {row.C:g} | {row.generated_trials} | {row.sample_f1:.4f} | {row.macro_f1:.4f} | {row.delta_vs_current_best:+.4f} |")
lines.extend(["",f"Best local configuration: **{best.experiment}**, sample-wise F1 **{best.sample_f1:.4f}**, macro F1 **{best.macro_f1:.4f}**.","",
    "All label confidence thresholds, per-reason budgets, training-only scalers, weight normalization, threshold 0.5, "
    "and nonempty fallback remain fixed. The central configuration reproduces the previous model's generated targets, weights, and probabilities. "
    "Original known labels are preserved, DBSCAN noise stays unknown for generated targets, and validation never enters fitting. "
    "The selected model is reloaded and independently refitted from its saved dataset with matching predictions.","",
    "This is a bounded local search selected using validation, not a proof of a global maximum or independent Kaggle improvement. "
    "Nearby density settings can produce identical label datasets. No Kaggle submission was used and the best-submitted tag remains unchanged.","",
    "`datasets/`, `label_audits/`, `clusters/`, `labeling_models/`, `classifier_models/`, and `validation_predictions/` "
    "preserve all inputs and outputs. `comparison.csv`, `density_comparison.csv`, `label_counts.csv`, and `settings.json` preserve "
    "the complete grid, diagnostics, settings, and checksums.",""])
if candidate:lines.append(f"New candidate prepared locally: `{candidate['submission_file']}`. Its Kaggle score is unknown.")
else:lines.append("The grid did not improve the primary validation metric; retain the existing DBSCAN candidate rather than creating a new submission file.")
summary="\n".join(lines).rstrip()+"\n"
(OUT/"SUMMARY.md").write_text(summary)
print(summary)

# Focused DBSCAN and logistic regression refinement

A fixed local grid evaluates 9 density settings and 3 classifier regularization strengths each, using only the same 150 embeddings. The previous DBSCAN + LR reference is sample-wise F1 0.9103 and macro F1 0.8114 on the same 78 validation trials.

Radius center: **0.00012142366141394104**. Radius factors: **0.5,1,2**. Min_samples: **3,5,8**. C: **100,1000,10000**.

| Min samples | Radius factor | C | Generated trials | Sample-wise F1 | Macro F1 | Change from reference |
|---|---:|---:|---:|---:|---:|---:|
| 5 | 1 | 1000 | 133 | 0.9103 | 0.8114 | +0.0000 |
| 3 | 1 | 1000 | 133 | 0.9103 | 0.8114 | +0.0000 |
| 8 | 1 | 1000 | 133 | 0.9103 | 0.8114 | +0.0000 |
| 5 | 0.5 | 1000 | 81 | 0.8974 | 0.8039 | -0.0128 |
| 5 | 2 | 1000 | 173 | 0.8974 | 0.8039 | -0.0128 |
| 5 | 1 | 10000 | 133 | 0.8974 | 0.8039 | -0.0128 |
| 5 | 0.5 | 10000 | 81 | 0.8974 | 0.8039 | -0.0128 |
| 5 | 2 | 10000 | 173 | 0.8974 | 0.8039 | -0.0128 |
| 3 | 0.5 | 1000 | 81 | 